# **Simple RAG System**

## **Downloading Data**

In [16]:
# Central Configuration Dictionary to manage all system parameters
config = {
    "data_dir": "../data",                           # Directory to store raw and cleaned data
    "vector_store_dir": "./vector_store",           # Directory to persist our vector store
    "llm_provider": "openai",                       # The LLM provider we are using
    "reasoning_llm": "gpt-4o",                      # The powerful model for planning and synthesis
    "fast_llm": "gpt-4o-mini",                      # A faster, cheaper model for simpler tasks like the baseline RAG
    "embedding_model": "text-embedding-3-small",    # The model for creating document embeddings
    "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2", # The model for precision reranking
    "max_reasoning_iterations": 7,                  # A safeguard to prevent the agent from getting into an infinite loop
    "top_k_retrieval": 10,                          # Number of documents for initial broad recall
    "top_n_rerank": 3,                              # Number of documents to keep after precision reranking
}

In [17]:
# importing all necessary libraries
from dotenv import load_dotenv
import os
import json
import uuid
import re
from pprint import pprint
from typing import List, Literal, Dict, Optional, TypedDict

# loading all environment variables
load_dotenv()

True

In [18]:
import requests
from bs4 import BeautifulSoup
    
# Custom function downloading the 10-K filing and parsing the raw HTML, and converting it into a clean and structured text format suitable for ingestion by RAG pipeline

def download_and_parse_10k(url, data_path_raw, data_path_clean):
    # check if cleaned url already exits to avoid redownloading of dataset
    if os.path.exists(data_path_clean):
        print(f'Cleaned data already exitst at {data_path_clean}')
        return 
    
    # downloading 10k-filling
    print(f"Downloading 10k-filing from url {url}")
    headers = {
    "User-Agent": "MyRAGResearchBot/1.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    }
    respose = requests.get(url=url, headers=headers)
    respose.raise_for_status()

    with open(data_path_raw, 'w', encoding='utf-8') as f:
        f.write(respose.text)
    print(f"Raw data saved to {data_path_raw}")

    soup = BeautifulSoup(respose.content, 'html.parser')

    final_text = []
    for i in soup.find_all(['p', 'div', 'span']):
        text = i.get_text(strip=True) + '\n\n'
        # normalized = re.sub(r"\s+", " ", text)

        if final_text and text == final_text[-1]:
            continue

        final_text.append(text) 

    text = ' '.join(final_text)

    clean_text = re.sub(r'\n{3,}', '\n\n', text).strip()
    clean_text = re.sub(r'[ \t]{2,}', ' ', clean_text).strip()

    with open(data_path_clean, 'w', encoding='utf-8') as f:
        f.write(clean_text)
    print(f'Cleaned text content extracted and saved to {data_path_clean}')

In [19]:
# Execute the download and parsing function

# URL for NVIDIA's 2023 10-K filing (filed Feb 2023 for fiscal year ending Jan 2023)
url_10k = "https://www.sec.gov/Archives/edgar/data/1045810/000104581023000017/nvda-20230129.htm"
os.makedirs(config["data_dir"], exist_ok=True)
data_path_raw = os.path.join(config["data_dir"], "nvda_10k_2023_raw.html")
data_path_clean = os.path.join(config["data_dir"], "nvda_10k_2023_clean.txt")

print("Downloading and parsing NVIDIA's 2023 10-K filing...")
download_and_parse_10k(url_10k, data_path_raw, data_path_clean)

with open(data_path_clean, 'r', encoding='utf-8') as f:
    print("--- Sample content from cleaned 10-K ---")
    print(f.read(1000) + "...")

Cleaned data already exitst at ../data\nvda_10k_2023_clean.txt
--- Sample content from cleaned 10-K ---
00010458102023FYfalseP3YP4YP5YP1YP3YP1Yhttp://fasb.org/us-gaap/2022#AccruedLiabilitiesCurrenthttp://fasb.org/us-gaap/2022#AccruedLiabilitiesCurrenthttp://fasb.org/us-gaap/2022#AccruedLiabilitiesCurrent00010458102022-01-312023-01-2900010458102022-07-29iso4217:USD00010458102023-02-17xbrli:shares00010458102021-02-012022-01-3000010458102020-01-272021-01-31iso4217:USDxbrli:shares00010458102023-01-2900010458102022-01-300001045810us-gaap:CommonStockMember2020-01-260001045810us-gaap:AdditionalPaidInCapitalMember2020-01-260001045810us-gaap:TreasuryStockMember2020-01-260001045810us-gaap:AccumulatedOtherComprehensiveIncomeMember2020-01-260001045810us-gaap:RetainedEarningsMember2020-01-2600010458102020-01-260001045810us-gaap:RetainedEarningsMember2020-01-272021-01-310001045810us-gaap:AccumulatedOtherComprehensiveIncomeMember2020-01-272021-01-310001045810us-gaap:CommonStockMember2020-01-272021-01

## **Building Simple RAG Application**

In [20]:
# importing required libraries
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# loadin text data
print('Loading and chunking documents...')
loader = TextLoader(data_path_clean, encoding='utf-8')
docs = loader.load()

# splitting text data into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
splitted_docs = text_splitter.split_documents(docs)

print(f"Documents loaded and split into {len(splitted_docs)} chunks.")


Loading and chunking documents...
Documents loaded and split into 571 chunks.


In [21]:
# creating embeddings and vector store

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma

print('Creating baseline vector store...')
# creating embedding model
embeddings = OpenAIEmbeddings(model=config['embedding_model'])

# creating vector store
vector_store = Chroma.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

# creating retriever
baseline_retriever = vector_store.as_retriever(search_kwargs={'k':3})

print(f'Vector store created with {vector_store._collection.count()} embeddings')

Creating baseline vector store...
Vector store created with 1142 embeddings


In [22]:
# creating prompt template

from langchain_core.prompts import ChatPromptTemplate

template = """You are an AI financial analyst. Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [23]:
# Creating RAG chain

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# string output parser
str_output_parser = StrOutputParser()

# defining LLM for answering question
llm = ChatOpenAI(model=config['fast_llm'])

# function for joining retrieved text
def join_retrieved_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# basic RAG chain
basic_rag_chain = (
    {'context': baseline_retriever | join_retrieved_docs, 'question': RunnablePassthrough()} 
    | prompt
    | llm
    | str_output_parser
)


In [24]:
# checking inference

from rich.console import Console # For pretty-printing output with markdown
from rich.markdown import Markdown

# Initialize the rich console for better output formatting
console = Console()

# Our complex, multi-hop, multi-source query
complex_query_adv = "Based on NVIDIA's 2023 10-K filing, identify their key risks related to competition. Then, find recent news (post-filing, from 2024) about AMD's AI chip strategy and explain how this new strategy directly addresses or exacerbates one of NVIDIA's stated risks."

print("Executing complex query on the baseline RAG chain...")
# Invoke the chain with our challenging query
baseline_result = basic_rag_chain.invoke(complex_query_adv)

console.print("\n--- BASELINE RAG FAILED OUTPUT ---")
# Print the result using markdown formatting for readability
console.print(Markdown(baseline_result))

Executing complex query on the baseline RAG chain...


--- BASELINE RAG FAILED OUTPUT ---

Based on NVIDIA's 2023 10-K filing, the key risks related to competition include:                                  

 1 Increased Competitive Environment: NVIDIA faces significant competition from companies that provide or intend to
   provide GPUs, CPUs, DPUs, and other accelerated computing products, particularly those catering to AI. This     
   competitive landscape is expected to intensify in the future.                                                   
 2 Resource Disparity: Some competitors may possess greater marketing, financial, distribution, and manufacturing  
   resources, enabling them to adapt more readily to technological changes or customer preferences compared to     
   NVIDIA.                                                                                                         
 3 Failure to Meet Evolving Needs: NVIDIA must continuously meet the evolving needs of its industry and markets to 
   maintain its financial results. Failing to do so could adversely impact its business.                           

Recent news from 2024 regarding AMD's AI chip strategy indicates that the company has aggressively invested in     
developing advanced AI chips aimed at providing competitive solutions in the AI market. AMD's strategy includes the
introduction of new architectures and leveraging significant partnerships to enhance their product offerings and   
market share in the AI domain.                                                                                     

This new strategy directly addresses NVIDIA's stated risk of increased competition. By advancing its AI chip       
technology, AMD enhances its position in the market, potentially capturing market share that could otherwise go to 
NVIDIA. This exacerbates NVIDIA's risk of not being able to maintain its financial results and competitive edge due
to the evolving nature of customer demands and technological advancements in the AI sector. As AMD positions itself
as a formidable competitor with innovative AI solutions, NVIDIA might face challenges in adapting quickly enough to
maintain its market leadership.

In [25]:
from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    AnswerCorrectness,
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import pandas as pd

print("Preparing evaluation dataset...")

ground_truth_answer_adv = (
    "NVIDIA's 2023 10-K lists intense competition and rapid technological change "
    "as key risks. This risk is exacerbated by AMD's 2024 strategy, specifically "
    "the launch of the MI300X AI accelerator, which directly competes with NVIDIA's "
    "H100 and has been adopted by major cloud providers, threatening NVIDIA's market "
    "share in the data center segment."
)

retrieved_docs_for_baseline_adv = baseline_retriever.invoke(complex_query_adv)
baseline_contexts = [doc.page_content for doc in retrieved_docs_for_baseline_adv]

samples = [
    SingleTurnSample(
        user_input=complex_query_adv,
        response=baseline_result,
        retrieved_contexts=baseline_contexts,
        reference=ground_truth_answer_adv,
    )
]

eval_dataset = EvaluationDataset(samples=samples)

# ------------------------------------------------------------
# Evaluator LLM with higher max_tokens to avoid incomplete output
# ------------------------------------------------------------
judge_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=2000,
)

evaluator_llm = LangchainLLMWrapper(judge_llm)

# ------------------------------------------------------------
# Embeddings needed by some Ragas metrics like AnswerCorrectness
# ------------------------------------------------------------
evaluator_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
metrics = [
    ContextPrecision(llm=evaluator_llm),
    ContextRecall(llm=evaluator_llm),
    Faithfulness(llm=evaluator_llm),
    AnswerCorrectness(llm=evaluator_llm, embeddings=evaluator_embeddings),
]

print("Running RAGAs evaluation...")

result = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
)

print("Evaluation complete.")

results_df = result.to_pandas()
results_df.index = ["baseline_rag"]

print("\n--- RAGAs Evaluation Results ---")
print(results_df)

expected_cols = [
    "context_precision",
    "context_recall",
    "faithfulness",
    "answer_correctness",
]
available_cols = [c for c in expected_cols if c in results_df.columns]

if available_cols:
    print("\n--- Selected Metrics ---")
    print(results_df[available_cols].T)
else:
    print("\nMetric columns differ in your Ragas version. Available columns are:")
    print(results_df.columns.tolist())

C:\Users\masan\AppData\Local\Temp\ipykernel_13152\1550947318.py:5: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\masan\AppData\Local\Temp\ipykernel_13152\1550947318.py:5: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import (
C:\Users\masan\AppData\Local\Temp\ipykernel_13152\1550947318.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\masan\AppData\Local\Temp\ipykernel_13152\1550

Preparing evaluation dataset...


C:\Users\masan\AppData\Local\Temp\ipykernel_13152\1550947318.py:47: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(judge_llm)
C:\Users\masan\AppData\Local\Temp\ipykernel_13152\1550947318.py:52: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(


Running RAGAs evaluation...


Evaluating: 100%|██████████| 4/4 [00:32<00:00,  8.16s/it]

Evaluation complete.

--- RAGAs Evaluation Results ---
                                                     user_input  \
baseline_rag  Based on NVIDIA's 2023 10-K filing, identify t...   

                                             retrieved_contexts  \
baseline_rag  [ITEM 1A. RISK FACTORS\n\n In evaluating NVIDI...   

                                                       response  \
baseline_rag  Based on NVIDIA's 2023 10-K filing, the key ri...   

                                                      reference  \
baseline_rag  NVIDIA's 2023 10-K lists intense competition a...   

              context_precision  context_recall  faithfulness  \
baseline_rag                1.0             0.5      0.619048   

              answer_correctness  
baseline_rag            0.896455  

--- Selected Metrics ---
                    baseline_rag
context_precision       1.000000
context_recall          0.500000
faithfulness            0.619048
answer_correctness      0.896455
